# Rerun with extract_final_v2 (22 Oct 2026)

Reruns, with fractions read in full (`extract_final_v2.py`), the analyses that used the as-run reader:
the flatness TOST and Qwen→Pro transition table, the k≥1–4 sweep, risk control (Table 4), exact matching,
and the Screen A/B cross-tabs. It ends with the scorer check on GSM-Hard's stored golds and, as decided on
23 Oct, GSM-Hard's matching-rule sensitivity check at the golds' stored precision (`match_stored_precision.py`),
which replaces exact matching. The primary scorer is unchanged (`stage2_lib.answers_match`).

- **Every analysis first reproduces its published as-run numbers with v1** (gates A–G). A failed gate stops the notebook.
- `stage2_lib.py` is loaded from an explicit path and never changed. `ckpt\` is read only.
- Only rows that v2 reads differently are re-verified with Z3, with the v2 value as the target.
  The same rows re-verified with the v1 target must reproduce the stored verdicts first.
- CPU only. Writes only `out\rerun_v2\<timestamp>\`.
- Run from `Videos\tost flatness`, or set the paths in the first code cell.

In [ ]:
# =============================================================================
# CELL 1 - PATHS AND SETTINGS
# =============================================================================
import csv, datetime as dt, hashlib, importlib.util, json, math, random, re, sys, time
from collections import Counter, defaultdict
from decimal import Decimal, ROUND_HALF_EVEN, localcontext
from fractions import Fraction as F
from math import comb
from pathlib import Path

ROOT = Path.cwd()                                    # Videos\tost flatness
CKPT = ROOT / "ckpt"
STAGE2 = ROOT / "stage2_lib.py"                      # the as-run copy
V2 = ROOT / "extract_final_v2.py"
MSP = ROOT / "match_stored_precision.py"             # GSM-Hard sensitivity scorer (decided 23 Oct)
TOST_NB = ROOT / "tost_flatness.ipynb"               # its cells 2, 4 and 5 are run verbatim
TOST_PUBLISHED = ROOT / "out" / "tost_flatness" / "20260927_234744" / "tost_results.json"
C3_DIR = ROOT / "c3_extraction_output" / "extraction_audit_20261022_202416"   # optional cross-check
FLASHLITE_DIR = Path(r"C:\Users\Abhis\Videos\Audit\out\google_api_audit\20260917_162638")
SCAN_DIR = Path(r"C:\Users\Abhis\Videos\direction B 2\files\gsm_hard_scan_out")

TIMEOUT_MS = 10000          # the pipeline's default
FULL_GATE = True            # also re-verify all of Qwen SVAMP (4,000 calls, under a minute)

AS_RUN_SHA256_LF = "ae042d0c48c9d0f088ade4fc718ba5b56fc37328c9b0e47ca1306f2c20941388"
DATASETS = ["svamp", "gsm8k", "gsm_hard", "gsm_symbolic_main", "gsm_symbolic_p1", "gsm_symbolic_p2"]
SYMBOLIC = {"gsm_symbolic_main", "gsm_symbolic_p1", "gsm_symbolic_p2"}
GENS = ["qwen", "flashlite", "pro"]
GEN_FILES = {"qwen": "raw_gen_{ds}.jsonl", "flashlite": "raw_gen_flashlite_{ds}.jsonl",
             "pro": "raw_gen_gemini_{ds}.jsonl"}
N_EXPECTED = {"svamp": 1000, "gsm8k": 1319, "gsm_hard": 1319,
              "gsm_symbolic_main": 5000, "gsm_symbolic_p1": 5000, "gsm_symbolic_p2": 2500}
EXCLUDED = {"gsm_symbolic_main": {76}, "gsm_symbolic_p1": {76}}
M4 = ["t1", "t2", "t3", "t6"]

for p in (CKPT, STAGE2, V2, MSP, TOST_NB, FLASHLITE_DIR, SCAN_DIR):
    assert p.exists(), f"not found: {p}"
RUN_TS = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "out" / "rerun_v2" / RUN_TS
assert CKPT.resolve() not in OUT.resolve().parents
print("root:", ROOT, "\noutputs:", OUT)

In [ ]:
# =============================================================================
# CELL 2 - LOAD THE AS-RUN stage2_lib AND extract_final_v2 FROM EXPLICIT PATHS
# =============================================================================
def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


with open(STAGE2, "rb") as fh:
    s2_sha = hashlib.sha256(fh.read().replace(b"\r\n", b"\n")).hexdigest()
assert s2_sha == AS_RUN_SHA256_LF, f"{STAGE2} is not the as-run stage2_lib.py (sha256 {s2_sha[:16]})"
s2 = load_module(STAGE2, "stage2_lib_as_run")
v2 = load_module(V2, "extract_final_v2")
msp = load_module(MSP, "match_stored_precision")
n_msp = msp._self_test()
import z3
print("stage2_lib: as-run copy (sha256 of LF text", s2_sha[:16] + ")")
print(v2.VERSION, "|", msp.VERSION, f"(self-test {n_msp} cases ok)", "| z3", z3.get_version_string(),
      "| python", sys.version.split()[0])

In [ ]:
# =============================================================================
# CELL 3 - PER-ROW STATES AS RUN (v1). GATE A: the stored verdicts and the v1
# reading reproduce every cell of arms_three_generators.json (N, E, A, W, T).
# =============================================================================
def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


def text_of(r):
    o = r.get("outputs")
    return (o[0] if o else "") if isinstance(o, list) else (o or "")


def template_of(pid):
    tail = pid.split("/")[1]
    return int(tail.split("_")[0]) if "_" in tail else None


def verdict(v):
    """Bools, except Flash-Lite rows verified on 17 Sep, which store verify()'s dict."""
    if isinstance(v, bool):
        return v
    if isinstance(v, dict):
        return bool(v.get("certified", False))
    raise TypeError(repr(v)[:80])


def cert_block(gen, ds):
    if gen == "flashlite":
        return read_json(FLASHLITE_DIR / f"w2s_cert_flashlite_{ds}.json")["flashlite"]
    if ds in SYMBOLIC:
        return read_json(CKPT / f"w2s_cert_{ds}.json")["qwen" if gen == "qwen" else "gemini"]
    if gen == "pro":
        d = read_json(CKPT / f"w2s_cert_{ds}.json")
    else:
        d = read_json(CKPT / f"_verdict_cache_{ds}.json")
    return {"correct": d["correct"], "cert": d["cert"]}


STATES = {}
for ds in DATASETS:
    for gen in GENS:
        recs = read_jsonl(CKPT / GEN_FILES[gen].format(ds=ds))
        assert len(recs) == N_EXPECTED[ds] and len({r["pid"] for r in recs}) == len(recs), (gen, ds)
        assert not any(r.get("ok") is False for r in recs), (gen, ds, "ok=False rows")
        blk = cert_block(gen, ds)
        rows = {}
        for r in recs:
            pid, t = r["pid"], text_of(r)
            gold = s2.to_fraction(r.get("gold"))
            e1, tag = s2.extract_final(t)
            e2, tag2 = v2.extract_final_v2(t)
            assert tag == tag2, (gen, pid)
            rows[pid] = {"gold_str": r.get("gold"), "gold": gold, "e1": e1, "e2": e2, "tag": tag,
                         "ok1": s2.answers_match(e1, gold), "ok2": s2.answers_match(e2, gold),
                         "cert1": {t_: verdict(blk["cert"][t_][pid]) for t_ in blk["cert"]},
                         "template": template_of(pid),
                         "eligible": ds not in EXCLUDED or template_of(pid) not in EXCLUDED[ds]}
            assert blk["correct"][pid] == rows[pid]["ok1"], (gen, pid, "stored correct != v1 rescoring")
        STATES[(gen, ds)] = rows

PAIRS_P1 = [(a, b) for i, a in enumerate(M4) for b in M4[i + 1:]]
SIX = ["t1", "t2", "t3", "t4", "t5", "t6"]
PAIRS_SYM = [(a, b) for i, a in enumerate(SIX) for b in SIX[i + 1:]]


def rules_for(ds):
    rules = [(f"k>={k} m4", M4, k) for k in (1, 2, 3, 4)]
    rules += [(f"AND {a}x{b}", [a, b], 2) for a, b in (PAIRS_SYM if ds in SYMBOLIC else PAIRS_P1)]
    if ds in SYMBOLIC:
        rules += [(f"k>={k} m5", M4 + ["t4"], k) for k in range(1, 6)]
        rules += [(f"k>={k} m6", M4 + ["t4", "t5"], k) for k in range(1, 7)]
    return rules


def arm(ds, rows, members, k, v):
    N = E = A = W = 0
    T = set()
    for r in rows.values():
        if not r["eligible"]:
            continue
        N += 1
        ok = r[f"ok{v}"]
        E += not ok
        if sum(r[f"cert{v}"][t] for t in members) >= k:
            A += 1
            W += not ok
            T.add(r["template"])
    return {"N": N, "E": E, "A": A, "W": W, "T": len(T) if ds in SYMBOLIC else None}


ARMS_PUB = read_json(FLASHLITE_DIR / "arms_three_generators.json")
GEN_KEY = {"qwen": "qwen", "flashlite": "flashlite", "pro": "gemini"}
bad = []
n_cells = 0
for ds in DATASETS:
    for gen in GENS:
        for name, members, k in rules_for(ds):
            got, pub = arm(ds, STATES[(gen, ds)], members, k, 1), ARMS_PUB[ds][name][GEN_KEY[gen]]
            n_cells += 1
            if any(got[x] != pub[x] for x in ("N", "E", "A", "W")) or (ds in SYMBOLIC and got["T"] != pub["T"]):
                bad.append((ds, gen, name, got, pub))
assert not bad, f"GATE A FAILED on {len(bad)} cells, e.g. {bad[:3]}"
print(f"GATE A passed: stored correct maps equal the v1 rescoring on all 18 files, and "
      f"{n_cells} published arms cells (N, E, A, W, T) are reproduced")

In [ ]:
# =============================================================================
# CELL 4 - v2 READINGS. Re-verify every row v2 reads differently, with Z3.
# GATE B: those rows are exactly C3's flagged rows (if C3's output is present).
# GATE C: re-verified with the v1 target they reproduce the stored verdicts
#         (and, with FULL_GATE, all of Qwen SVAMP does too: 560 / 4).
# =============================================================================
CHANGED = [(gen, ds, pid) for (gen, ds), rows in STATES.items() for pid, r in rows.items() if r["e1"] != r["e2"]]
print("rows v2 reads differently:", len(CHANGED), dict(Counter((g, d) for g, d, _ in CHANGED)))
if (C3_DIR / "flagged_rows.csv").exists():
    with open(C3_DIR / "flagged_rows.csv", encoding="utf-8-sig") as fh:
        c3 = {(r["generator"], r["pid"]): r for r in csv.DictReader(fh) if r["status"] == "disagree"}
    assert set(c3) == {(g, p) for g, d, p in CHANGED}, "GATE B FAILED: rows differ from C3's flagged rows"
    for g, d, p in CHANGED:
        assert F(c3[(g, p)]["careful"]) == STATES[(g, d)][p]["e2"], ("GATE B FAILED", g, p)
    print(f"GATE B passed: the {len(CHANGED)} rows are C3's flagged rows, with C3's careful values")
else:
    print("GATE B skipped (no C3 output folder)")

_TRANS = {}


def trans_text(t, ds, pid):
    if (t, ds) not in _TRANS:
        recs = read_jsonl(CKPT / f"raw_{t}_{ds}.jsonl")
        assert len(recs) == N_EXPECTED[ds] and len({r["pid"] for r in recs}) == len(recs), (t, ds)
        _TRANS[(t, ds)] = {r["pid"]: text_of(r) for r in recs}
    return _TRANS[(t, ds)][pid]


t0 = time.time()
if FULL_GATE:
    rows = STATES[("qwen", "svamp")]
    mism = [(t, p) for p, r in rows.items() for t in M4
            if s2.verify(trans_text(t, "svamp", p), r["e1"], TIMEOUT_MS)["certified"] != r["cert1"][t]]
    A = sum(sum(r["cert1"][t] for t in M4) >= 3 for r in rows.values())
    assert not mism, f"GATE C FAILED: {len(mism)} Qwen SVAMP verdicts differ, e.g. {mism[:5]}"
    print(f"GATE C (full): all 4,000 Qwen SVAMP verdicts reproduced ({time.time() - t0:.0f}s)")

REVERIFY = {}
mism = []
for gen, ds, pid in CHANGED:
    r = STATES[(gen, ds)][pid]
    r["cert2"] = {}
    REVERIFY[f"{gen}|{pid}"] = {}
    for t in r["cert1"]:
        text = trans_text(t, ds, pid)
        o1 = s2.verify(text, r["e1"], TIMEOUT_MS)
        o2 = s2.verify(text, r["e2"], TIMEOUT_MS)
        if o1["certified"] != r["cert1"][t]:
            mism.append((gen, pid, t))
        r["cert2"][t] = bool(o2["certified"])
        REVERIFY[f"{gen}|{pid}"][t] = {"v1_target": str(r["e1"]), "v1": o1, "v2_target": str(r["e2"]), "v2": o2}
assert not mism, f"GATE C FAILED: {len(mism)} changed-row verdicts differ from the stored ones, e.g. {mism[:5]}"
for rows in STATES.values():
    for r in rows.values():
        r.setdefault("cert2", dict(r["cert1"]))
print(f"GATE C passed: the changed rows' stored verdicts are reproduced; "
      f"{sum(len(v) for v in REVERIFY.values())} verdicts recomputed with the v2 target ({time.time() - t0:.0f}s)")

In [ ]:
# =============================================================================
# CELL 5 - EVERY ARMS CELL THAT MOVES, AND THE k>=1-4 SWEEP OVER m4
# =============================================================================
pct = lambda w, a: 100 * w / a if a else float("nan")
ARMS = {(ds, name, gen): (arm(ds, STATES[(gen, ds)], m, k, 1), arm(ds, STATES[(gen, ds)], m, k, 2))
        for ds in DATASETS for gen in GENS for name, m, k in rules_for(ds)}
MOVED = [(ds, gen, name, a1, a2) for (ds, name, gen), (a1, a2) in ARMS.items()
         if (a1["E"], a1["A"], a1["W"]) != (a2["E"], a2["A"], a2["W"])]
print("cells where E, A or W move (v1 -> v2):")
for ds, gen, name, a1, a2 in MOVED:
    print(f"  {ds:<16} {gen:<9} {name:<10} E {a1['E']}->{a2['E']}  A {a1['A']}->{a2['A']}  W {a1['W']}->{a2['W']}")
print("nothing moves on GSM-Symbolic:", not any(ds in SYMBOLIC for ds, *_ in MOVED))

SWEEP = []
print("\nk-sweep over m4, CW % (v1 -> v2 where it moves):")
for k in (1, 2, 3, 4):
    name = f"k>={k} m4"
    for v in (1, 2):
        gaps, spreads = [], []
        for ds in DATASETS:
            rate = {g: pct(ARMS[(ds, name, g)][v - 1]["W"], ARMS[(ds, name, g)][v - 1]["A"]) for g in GENS}
            gaps.append((abs(rate["pro"] - rate["qwen"]), ds, rate["qwen"], rate["pro"]))
            spreads.append((max(rate.values()) - min(rate.values()), ds))
            SWEEP.append({"rule": name, "reader": f"v{v}", "dataset": ds,
                          **{f"{g}_{x}": ARMS[(ds, name, g)][v - 1][x] for g in GENS for x in ("A", "W")}})
        g_ = max(gaps)
        print(f"  {name} v{v}: largest Qwen->Pro movement {g_[0]:.3f} pts ({g_[1]}: {g_[2]:.3f} -> {g_[3]:.3f}); "
              f"Pro below Qwen on {sum(x[3] < x[2] for x in gaps)}/6; largest 3-generator spread "
              f"{max(spreads)[0]:.3f} ({max(spreads)[1]})")

In [ ]:
# =============================================================================
# CELL 6 - TOST AND THE QWEN -> PRO TRANSITION TABLE: tost_flatness.ipynb's
# cells 2, 4 and 5 run verbatim. GATE D: v1 is identical to run 20260927_234744.
# =============================================================================
TOSTNB = read_json(TOST_NB)
tost_cell = lambda i: "".join(TOSTNB["cells"][i]["source"])
TOST_SETUP = '''
import csv, json, math, re, sys
import datetime as dt
from collections import Counter
from pathlib import Path
M4, K = ("t1", "t2", "t3", "t6"), 3
ALPHA = 0.05
MARGINS_PP = (1, 2, 3, 5)
UNCLUSTERED = ("svamp", "gsm8k", "gsm_hard")
SYMBOLIC = ("gsm_symbolic_main", "gsm_symbolic_p1", "gsm_symbolic_p2")
DATASETS = UNCLUSTERED + SYMBOLIC
EXCLUDE = {"gsm_symbolic_main": {76}, "gsm_symbolic_p1": {76}}
BELOW_FLOOR = {"gsm_symbolic_p2"}
def template_of(pid):
    tail = str(pid).rsplit("/", 1)[-1]
    parts = re.split(r"[_\\-]", tail)
    if len(parts) >= 2 and parts[-2].isdigit():
        return int(parts[-2])
    return int(re.search(r"\\d+", tail).group(0))
def population(ds, pids, which):
    if which == "table" and ds in EXCLUDE:
        return sorted(p for p in pids if template_of(p) not in EXCLUDE[ds])
    return sorted(pids)
def summarize(ds, rows, pop):
    A = W = E = 0
    tA = Counter()
    for p in pop:
        acc, ok = rows[p]
        E += not ok
        if acc:
            A += 1
            W += not ok
            if ds in SYMBOLIC:
                tA[template_of(p)] += 1
    return {"N": len(pop), "A": A, "W": W, "E": E, "T": len(tA) if ds in SYMBOLIC else None}
'''


def run_tost(v):
    g = {}
    exec(TOST_SETUP, g)
    exec(tost_cell(2), g)                                  # statistics + their own known-value checks
    g["ROWS"] = {ds: {arm_: {p: (sum(r[f"cert{v}"][t] for t in M4) >= 3, bool(r[f"ok{v}"]))
                             for p, r in STATES[(gen, ds)].items()}
                      for arm_, gen in (("qwen", "qwen"), ("gemini", "pro"))} for ds in DATASETS}
    exec('SUM = {ds: {a: {w: summarize(ds, ROWS[ds][a], population(ds, ROWS[ds][a], w)) for w in ("table", "all")}'
         ' for a in ("qwen", "gemini")} for ds in DATASETS}', g)
    g["ROUTE"] = {(ds, a): f"rerun_v2 states, reader v{v}" for ds in DATASETS for a in ("qwen", "gemini")}
    g["RUN_TS"] = f"{RUN_TS}_reader_v{v}"
    g["OUT"] = OUT / f"tost_reader_v{v}"
    print(f"\n######## reader v{v} ########")
    exec(tost_cell(4), g)
    exec(tost_cell(5), g)
    return g


T1 = run_tost(1)
if TOST_PUBLISHED.exists():
    pub = read_json(TOST_PUBLISHED)
    close = lambda a, b: (abs(a - b) <= 1e-9 * max(1, abs(b))) if isinstance(b, float) and a is not None else a == b
    diffs = [(ds, a, w) for ds in DATASETS for a in ("qwen", "gemini") for w in ("table", "all")
             if T1["SUM"][ds][a][w] != pub["counts"][ds][a][w]]
    diffs += [(ds, k) for ds in DATASETS for k, v_ in pub["joint"][ds].items() if not close(T1["JOINT"][ds][k], v_)]
    diffs += [(r["dataset"], k) for r, rp in zip(T1["RESULTS"], pub["results"]) for k, v_ in rp.items() if not close(r.get(k), v_)]
    assert not diffs, f"GATE D FAILED: {diffs[:5]}"
    print("\nGATE D passed: reader v1 reproduces run 20260927_234744 exactly (counts, transitions, every result field)")
else:
    print("\nGATE D skipped: published TOST run not found")
T2 = run_tost(2)

In [ ]:
# =============================================================================
# CELL 7 - RISK CONTROL (Table 4), Qwen arm. Functions copied verbatim from
# crc_core.py (public repo). GATE E: v1 reproduces crc_<ds>.json exactly
# (single split: selected k, calibration table, test; and the 500-resplit audits).
# =============================================================================
_Z = {0.100: 1.2816, 0.050: 1.6449, 0.025: 1.9600, 0.0167: 2.1279,
      0.0125: 2.2414, 0.010: 2.3263, 0.005: 2.5758, 0.001: 3.0902}


def _z_for(delta):
    return _Z[min(_Z, key=lambda d: abs(d - delta))]


def wilson_upper(errors, n, delta):
    if n == 0:
        return 1.0
    z = _z_for(delta)
    p = errors / n
    denom = 1 + z * z / n
    centre = p + z * z / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return min(1.0, (centre + half) / denom)


def rule_of_three(n):
    return 3.0 / n if n else 1.0


def agreement_counts(cert, roles, pids):
    return {p: sum(1 for r in roles if cert[r][p]) for p in pids}


def gated(counts, pids, k):
    return [p for p in pids if counts[p] >= k]


def risk_at_k(counts, correct, pids, k):
    cov = gated(counts, pids, k)
    err = sum(1 for p in cov if not correct[p])
    return len(cov), err, (err / len(cov) if cov else 0.0)


def select_k(counts, correct, cal_pids, alpha, delta, m, bound="wilson", bonferroni=True):
    per_k_delta = (delta / m) if bonferroni else delta
    ub = wilson_upper
    rows, chosen = [], None
    for k in range(1, m + 1):
        n, err, rate = risk_at_k(counts, correct, cal_pids, k)
        u = ub(err, n, per_k_delta)
        rows.append({"k": k, "n_cal": n, "err": err, "rate": rate, "upper": u, "qualifies": bool(u <= alpha)})
        if chosen is None and u <= alpha:
            chosen = k
    return chosen, {"per_k_delta": per_k_delta, "table": rows}


def evaluate(counts, correct, test_pids, k):
    n, err, rate = risk_at_k(counts, correct, test_pids, k)
    out = {"k": k, "n_test": len(test_pids), "n_covered": n,
           "coverage": n / len(test_pids) if test_pids else 0.0,
           "cw": err, "cw_rate": rate, "cw_upper_95": wilson_upper(err, n, 0.05)}
    if err == 0 and n:
        out["cw_rule_of_three_95"] = rule_of_three(n)
    return out


def audit(counts, correct, pids, alpha, delta, m, trials=500, seed=0, bound="wilson", bonferroni=True, cal_frac=0.5):
    rng = random.Random(seed)
    picks, rates, covs, viol, abstain = [], [], [], 0, 0
    for _ in range(trials):
        sh = pids[:]
        rng.shuffle(sh)
        cut = int(len(sh) * cal_frac)
        k, _ = select_k(counts, correct, sh[:cut], alpha, delta, m, bound, bonferroni)
        if k is None:
            abstain += 1
            continue
        picks.append(k)
        n, err, rate = risk_at_k(counts, correct, sh[cut:], k)
        if n == 0:
            continue
        rates.append(rate)
        covs.append(n / len(sh[cut:]))
        if rate > alpha:
            viol += 1
    return {"trials": trials, "abstained": abstain, "selected": len(picks),
            "k_distribution": dict(sorted(Counter(picks).items())), "violations": viol}


CRC = {}
for ds in ("svamp", "gsm8k", "gsm_hard"):
    pubc = read_json(CKPT / f"crc_{ds}.json")
    rows = STATES[("qwen", ds)]
    pids = sorted(rows)
    rng = random.Random(pubc["seed"])
    sh = pids[:]
    rng.shuffle(sh)
    cal, test = sh[:int(len(sh) * 0.5)], sh[int(len(sh) * 0.5):]
    for v in (1, 2):
        cert = {t: {p: rows[p][f"cert{v}"][t] for p in pids} for t in pubc["roles"]}
        correct = {p: rows[p][f"ok{v}"] for p in pids}
        counts = agreement_counts(cert, pubc["roles"], pids)
        for a, d in pubc["alphas"].items():
            k, diag = select_k(counts, correct, cal, float(a), pubc["delta"], 4, pubc["bound"], pubc["bonferroni"])
            ev = evaluate(counts, correct, test, k) if k else None
            au = audit(counts, correct, pids, float(a), pubc["delta"], 4, d["audit"]["trials"], pubc["seed"],
                       pubc["bound"], pubc["bonferroni"])
            CRC[(ds, a, v)] = {"k": k, "cal": [(r["n_cal"], r["err"]) for r in diag["table"]], "test": ev, "audit": au}
            if v == 1:
                ok = (k == d["selected_k"]
                      and ("calibration" not in d or CRC[(ds, a, 1)]["cal"] == [(r["n_cal"], r["err"]) for r in d["calibration"]["table"]])
                      and (not d.get("test") or (ev["n_covered"], ev["cw"]) == (d["test"]["n_covered"], d["test"]["cw"]))
                      and all(au[x] == d["audit"][x] for x in ("abstained", "selected", "violations"))
                      and {str(x): y for x, y in au["k_distribution"].items()} == {str(x): y for x, y in d["audit"]["k_distribution"].items()})
                assert ok, f"GATE E FAILED: {ds} alpha {a}"
print("GATE E passed: the single split and all 500-resplit audits of crc_svamp/gsm8k/gsm_hard.json are reproduced")
for (ds, a, v), c in CRC.items():
    if v == 2 and c != CRC[(ds, a, 1)]:
        c1 = CRC[(ds, a, 1)]
        f = lambda c: (f"k={c['k']} test {c['test']['n_covered']}/{c['test']['n_test']} "
                       f"({100 * c['test']['coverage']:.1f}%) CW {c['test']['cw']} ({100 * c['test']['cw_rate']:.2f}%)"
                       if c["test"] else "refuses")
        print(f"  {ds} alpha {a}: v1 {f(c1)}; audit {c1['audit']['violations']}/{c1['audit']['selected']} violations")
        print(f"  {' ' * len(ds)}          v2 {f(c)}; audit {c['audit']['violations']}/{c['audit']['selected']} violations, "
              f"k {c['audit']['k_distribution']}")

In [ ]:
# =============================================================================
# CELL 8 - GSM-HARD: EXACT MATCHING, NON-NEGATIVE INTEGER TARGETS, WRONG-SET
# OVERLAP. GATE F: v1 reproduces 31/320, 48/400, 21/271, 31/324, 256, 249.
# =============================================================================
GH = {g: STATES[(g, "gsm_hard")] for g in GENS}
GPIDS = sorted(GH["qwen"])
acc3 = lambda r, v: sum(r[f"cert{v}"][t] for t in M4) >= 3
exact = lambda e, g: e is not None and g is not None and e == g
EXACT = {}
for v in (1, 2):
    for g in GENS:
        A = [p for p in GPIDS if acc3(GH[g][p], v)]
        EXACT[(v, g)] = {"A": len(A), "W_scorer": sum(not GH[g][p][f"ok{v}"] for p in A),
                         "W_exact": sum(not exact(GH[g][p][f"e{v}"], GH[g][p]["gold"]) for p in A)}
nni = [p for p in GPIDS if GH["qwen"][p]["gold"].denominator == 1 and GH["qwen"][p]["gold"] >= 0]
NNI = {(v, g): (sum(not GH[g][p][f"ok{v}"] for p in nni if acc3(GH[g][p], v)), sum(acc3(GH[g][p], v) for p in nni))
       for v in (1, 2) for g in GENS}
WRONG = {v: {g: {p for p in GPIDS if not GH[g][p][f"ok{v}"]} for g in GENS} for v in (1, 2)}
ov = lambda v: (len(WRONG[v]["flashlite"] & WRONG[v]["pro"]),
                len(WRONG[v]["flashlite"] & WRONG[v]["pro"]) / len(WRONG[v]["flashlite"] | WRONG[v]["pro"]),
                len(WRONG[v]["flashlite"] & WRONG[v]["pro"] & WRONG[v]["qwen"]))
assert ((EXACT[(1, "qwen")]["W_exact"], EXACT[(1, "pro")]["W_exact"]) == (31, 48) and len(nni) == 900
        and NNI[(1, "qwen")] == (21, 271) and NNI[(1, "pro")] == (31, 324) and ov(1)[0] == 256 and ov(1)[2] == 249), "GATE F FAILED"
print("GATE F passed")
for v in (1, 2):
    e = {g: EXACT[(v, g)] for g in GENS}
    print(f"reader v{v}: exact-rational CW " + ", ".join(f"{g} {e[g]['W_exact']}/{e[g]['A']} ({pct(e[g]['W_exact'], e[g]['A']):.3f}%)" for g in GENS)
          + f"; Qwen->Pro gap {pct(e['pro']['W_exact'], e['pro']['A']) - pct(e['qwen']['W_exact'], e['qwen']['A']):.3f} pts")
    print(f"           integer targets (900): " + ", ".join(f"{g} {NNI[(v, g)][0]}/{NNI[(v, g)][1]} ({pct(*NNI[(v, g)]):.3f}%)" for g in GENS))
    print(f"           wrong sets: Flash-Lite ∩ Pro {ov(v)[0]} (Jaccard {ov(v)[1]:.3f}); wrong for all three {ov(v)[2]}")

In [ ]:
# =============================================================================
# CELL 9 - SCREEN A / B CROSS-TABS. Among each generator's wrong GSM-Hard answers:
# flagged rate among certified-wrong (k>=3 over m4) vs the other wrong answers;
# Fisher exact, one-sided (as recorded) and two-sided. GATE G: v1 reproduces
# Pro 25/37 vs 109/276 and Qwen 13/26 vs 122/492.
# =============================================================================
def _hyper(a, b, c, d):
    n1, n2, k = a + b, c + d, a + c
    lo, hi = max(0, k - n2), min(k, n1)
    den = comb(n1 + n2, k)
    return {x: F(comb(n1, x) * comb(n2, k - x), den) for x in range(lo, hi + 1)}


def fisher(a, b, c, d):
    pr = _hyper(a, b, c, d)
    greater = float(sum(p for x, p in pr.items() if x >= a))
    two = float(sum(p for x, p in pr.items() if p <= pr[a] * F(10**7 + 1, 10**7)))
    return greater, min(1.0, two)


def load_flags(path, key):
    recs = read_jsonl(path)
    assert len(recs) == 1319 and len({r["pid"] for r in recs}) == 1319, path
    return {r["pid"]: bool(r[key]) for r in recs}


SCREENS = {"A": load_flags(SCAN_DIR / "scan_results.jsonl", "flagged"),
           "B": load_flags(SCAN_DIR / "scan_results_B.jsonl", "flagged_b")}
XT = []
for v in (1, 2):
    for sc in ("A", "B"):
        for g in GENS:
            wrong = [p for p in GPIDS if not GH[g][p][f"ok{v}"]]
            cw = [p for p in wrong if acc3(GH[g][p], v)]
            oth = [p for p in wrong if not acc3(GH[g][p], v)]
            a, c = sum(SCREENS[sc][p] for p in cw), sum(SCREENS[sc][p] for p in oth)
            p1, p2 = fisher(a, len(cw) - a, c, len(oth) - c)
            XT.append({"reader": f"v{v}", "screen": sc, "generator": g, "cw_flagged": a, "cw": len(cw),
                       "other_flagged": c, "other_wrong": len(oth), "p_one_sided": p1, "p_two_sided": p2})
x = {(r["reader"], r["screen"], r["generator"]): r for r in XT}
assert ((x[("v1", "B", "pro")]["cw_flagged"], x[("v1", "B", "pro")]["cw"], x[("v1", "B", "pro")]["other_flagged"], x[("v1", "B", "pro")]["other_wrong"]) == (25, 37, 109, 276)
        and (x[("v1", "B", "qwen")]["cw_flagged"], x[("v1", "B", "qwen")]["cw"], x[("v1", "B", "qwen")]["other_flagged"], x[("v1", "B", "qwen")]["other_wrong"]) == (13, 26, 122, 492)
        and round(x[("v1", "B", "pro")]["p_one_sided"], 4) == 0.0011 and round(x[("v1", "B", "qwen")]["p_one_sided"], 4) == 0.0063), "GATE G FAILED"
print("GATE G passed (one-sided p 0.0011 and 0.0063 reproduced)")
for r in XT:
    print(f"  {r['reader']} Screen {r['screen']} {r['generator']:<9} CW {r['cw_flagged']}/{r['cw']} "
          f"({pct(r['cw_flagged'], r['cw']):.1f}%) vs other wrong {r['other_flagged']}/{r['other_wrong']} "
          f"({pct(r['other_flagged'], r['other_wrong']):.1f}%)  p one-sided {r['p_one_sided']:.4f}, two-sided "
          f"{r['p_two_sided']:.4f}{'  < 0.05/6' if r['p_one_sided'] < 0.05 / 6 else ''}")

In [ ]:
# =============================================================================
# CELL 10 - SCORER CHECK. GSM-Hard golds are program outputs stored as
# float(round(x, 10)), so they carry 10-decimal rounding and float noise.
#   S1 as run : answers_match (exact, else |p-g| <= 1e-6 |g|)
#   S2 exact  : p == g as rationals (the 'exact matching' analysis, superseded 23 Oct)
#   R10       : p rounds to the stored gold at 10 decimal places
#   S3 stored precision: match_stored_precision.answers_match_stored,
#              |p-g| <= 5e-11 + 1e-14 |g| (10-dp rounding plus float noise)
# The primary scorer is S1. S3 is the sensitivity check (Cell 11).
# =============================================================================
Q10 = Decimal("1e-10")


def r10(x):
    with localcontext() as ctx:
        ctx.prec = 80
        return (Decimal(x.numerator) / Decimal(x.denominator)).quantize(Q10, rounding=ROUND_HALF_EVEN)


S1 = lambda p, g: s2.answers_match(p, g)
S2 = lambda p, g: p is not None and g is not None and p == g
R10 = lambda p, g: p is not None and g is not None and r10(p) == r10(g)
S3 = lambda p, g, rel=msp.REL_TOL: msp.answers_match_stored(p, g, rel_tol=rel)
bad = [(gen, ds, pid) for (gen, ds), rows in STATES.items() if ds != "gsm_hard" for pid, r in rows.items()
       for v in (1, 2) if S3(r[f"e{v}"], r["gold"]) != S1(r[f"e{v}"], r["gold"])]
assert not bad, f"S3 and S1 disagree outside GSM-Hard: {bad[:5]}"
print("outside GSM-Hard, the stored-precision match agrees with answers_match on every row of every file")
SC = []
for ds in DATASETS:
    for g in GENS:
        for v in (1, 2):
            el = [(pid, r) for pid, r in STATES[(g, ds)].items() if r["eligible"]]
            tiny = sum(1 for _, r in el if r["gold"] != 0 and abs(r["gold"]) < F(5, 10**5))
            fw = [pid for pid, r in el if not S1(r[f"e{v}"], r["gold"]) and R10(r[f"e{v}"], r["gold"])]
            loose = [pid for pid, r in el if S1(r[f"e{v}"], r["gold"]) and not S3(r[f"e{v}"], r["gold"])]
            art = [pid for pid, r in el if not S2(r[f"e{v}"], r["gold"]) and S3(r[f"e{v}"], r["gold"])]
            cert_at = lambda pids, k: [p for p in pids if sum(STATES[(g, ds)][p][f"cert{v}"][t] for t in M4) >= k]
            SC.append({"dataset": ds, "generator": g, "reader": f"v{v}", "tiny_golds": tiny,
                       "wrong_but_right_at_10dp": len(fw), "of_which_certified_k1_k2_k3_k4": [len(cert_at(fw, k)) for k in (1, 2, 3, 4)],
                       "right_only_by_tolerance": len(loose), "of_which_certified_k3": cert_at(loose, 3),
                       "exact_mismatch_within_storage_precision": len(art), "rows_wrong_but_right_at_10dp": fw})
print("non-zero rows (every other dataset has integer or short-decimal golds and shows zeros):")
for s in SC:
    if s["wrong_but_right_at_10dp"] or s["right_only_by_tolerance"] or s["exact_mismatch_within_storage_precision"]:
        print(f"  {s['dataset']} {s['generator']:<9} {s['reader']}: tiny golds {s['tiny_golds']}; wrong but right at 10 dp "
              f"{s['wrong_but_right_at_10dp']} (certified k>=1..4: {s['of_which_certified_k1_k2_k3_k4']}); right only by the "
              f"1e-6 tolerance {s['right_only_by_tolerance']} (certified at k>=3: {s['of_which_certified_k3']}); exact "
              f"mismatches within storage precision {s['exact_mismatch_within_storage_precision']}")
SCORERS = {}
for v in (1, 2):
    for g in GENS:
        A = [r for r in GH[g].values() if acc3(r, v)]
        SCORERS[(v, g)] = {n: sum(not f(r[f"e{v}"], r["gold"]) for r in A) for n, f in (("S1", S1), ("S2", S2), ("S3", S3))}
        SCORERS[(v, g)]["A"] = len(A)
        SCORERS[(v, g)]["acc"] = {n: 100 * sum(f(r[f"e{v}"], r["gold"]) for r in GH[g].values()) / 1319
                                  for n, f in (("S1", S1), ("S2", S2), ("S3", S3))}
g_ = {"ALPHA": 0.05}
exec("import math\n" + tost_cell(2).split("# ---- known-value checks")[0], g_)
print("\nGSM-Hard, k>=3 over m4, certified-wrong under each scorer:")
for v in (1, 2):
    for sc in ("S1", "S2", "S3"):
        wq, aq, wp, ap = SCORERS[(v, "qwen")][sc], SCORERS[(v, "qwen")]["A"], SCORERS[(v, "pro")][sc], SCORERS[(v, "pro")]["A"]
        lo, hi = g_["mn_ci"](wp, ap, wq, aq)
        print(f"  v{v} {sc}: " + ", ".join(f"{g} {SCORERS[(v, g)][sc]}/{SCORERS[(v, g)]['A']}" for g in GENS)
              + f"; Qwen->Pro {100 * (wp / ap - wq / aq):+.3f} pts; MN 90% [{100 * lo:+.2f}, {100 * hi:+.2f}]; "
                f"TOST p at 5 pp {g_['tost_p_mn'](wp, ap, wq, aq, 0.05):.3f}")
# How much float-noise allowance does each GSM-Hard answer need to match? The rule is safe
# only if REL_TOL sits above every noise case and below every answer that really differs.
need = []
for g in GENS:
    for pid, r in GH[g].items():
        for v in (1, 2):
            e, gold = r[f"e{v}"], r["gold"]
            if e is not None and gold not in (None, 0) and abs(e - gold) > msp.ABS_TOL:
                need.append(((abs(e - gold) - msp.ABS_TOL) / abs(gold), g, pid, v))
noise = max(x for x in need if x[0] <= msp.REL_TOL)
above = min(x for x in need if x[0] > msp.REL_TOL)
assert noise[0] < msp.REL_TOL < above[0]
print(f"REL_TOL {float(msp.REL_TOL):.0e}: float noise needs at most {float(noise[0]):.1e} ({noise[1]} {noise[2]}); "
      f"the first answer that differs needs {float(above[0]):.1e} ({above[1]} {above[2]}), so any allowance in between "
      f"gives the same verdict on every row")
for rel in (F(1, 10**14), F(1, 10**12), F(1, 10**9)):
    for v in (1, 2):
        for g in GENS:
            for k in (1, 2, 3, 4):
                A = [r for r in GH[g].values() if sum(r[f"cert{v}"][t] for t in M4) >= k]
                assert sum(not S3(r[f"e{v}"], r["gold"], rel) for r in A) == sum(not S3(r[f"e{v}"], r["gold"]) for r in A)
print("certified-wrong counts (k>=1-4, both readers) are also identical for any allowance from 1e-14 to 1e-9")

In [ ]:
# =============================================================================
# CELL 11 - GSM-HARD SENSITIVITY CHECK AT THE GOLDS' STORED PRECISION
# (decided 23 Oct; replaces exact matching). The certified set is the
# verifier's and does not change: only which certified answers count as wrong.
# =============================================================================
MS = lambda e, g: msp.answers_match_stored(e, g)
STORED = []
for v in (1, 2):
    for g in GENS:
        rows = GH[g]
        right_p = sum(r[f"ok{v}"] for r in rows.values())
        right_s = sum(MS(r[f"e{v}"], r["gold"]) for r in rows.values())
        for k in (1, 2, 3, 4):
            A = [r for r in rows.values() if sum(r[f"cert{v}"][t] for t in M4) >= k]
            STORED.append({"reader": f"v{v}", "generator": g, "rule": f"k>={k} m4", "A": len(A),
                           "W_primary": sum(not r[f"ok{v}"] for r in A),
                           "W_stored": sum(not MS(r[f"e{v}"], r["gold"]) for r in A),
                           "right_primary": right_p, "right_stored": right_s})
ST = {(s["reader"], s["generator"], s["rule"]): s for s in STORED}
for v in ("v1", "v2"):
    print(f"\n== reader {v}: GSM-Hard scored at the golds' stored precision (primary 1e-6 scorer in brackets) ==")
    for g in GENS:
        s = ST[(v, g, "k>=3 m4")]
        print(f"  {g:<9} accuracy {100 * s['right_stored'] / 1319:.2f}% [{100 * s['right_primary'] / 1319:.2f}%]; "
              f"k>=3 CW {s['W_stored']}/{s['A']} = {pct(s['W_stored'], s['A']):.3f}% "
              f"[{s['W_primary']}/{s['A']} = {pct(s['W_primary'], s['A']):.3f}%]")
    for k in (1, 2, 3, 4):
        q, f_, p_ = (ST[(v, g, f"k>={k} m4")] for g in ("qwen", "flashlite", "pro"))
        rates = [pct(x["W_stored"], x["A"]) for x in (q, f_, p_)]
        print(f"  k>={k}: Qwen->Pro {pct(p_['W_stored'], p_['A']) - pct(q['W_stored'], q['A']):+.3f} pts "
              f"[{pct(p_['W_primary'], p_['A']) - pct(q['W_primary'], q['A']):+.3f}]; 3-generator spread "
              f"{max(rates) - min(rates):.3f} pts")
    q, p_ = ST[(v, "qwen", "k>=3 m4")], ST[(v, "pro", "k>=3 m4")]
    lo, hi = g_["mn_ci"](p_["W_stored"], p_["A"], q["W_stored"], q["A"])
    ps = [g_["tost_p_mn"](p_["W_stored"], p_["A"], q["W_stored"], q["A"], m / 100) for m in (1, 2, 3, 5)]
    print(f"  k>=3 TOST: MN 90% [{100 * lo:+.2f}, {100 * hi:+.2f}], D_min {100 * max(-lo, hi):.2f}; "
          f"p at 1/2/3/5 pp {' / '.join(f'{x:.3f}' for x in ps)}")
    for g in GENS:
        A = [p for p in nni if acc3(GH[g][p], int(v[1]))]
        W = sum(not MS(GH[g][p][f"e{v[1]}"], GH[g][p]["gold"]) for p in A)
        print(f"  integer targets (900), {g}: {W}/{len(A)} = {pct(W, len(A)):.3f}%")

In [ ]:
# =============================================================================
# CELL 12 - WRITE THE OUTPUTS AND COMPARE WITH THE 22-23 OCT CONTAINER RUNS
# =============================================================================
OUT.mkdir(parents=True, exist_ok=True)


def jdump(obj, name):
    with open(OUT / name, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=1, default=str)


def cdump(rows, name):
    with open(OUT / name, "w", encoding="utf-8", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(dict.fromkeys(k for r in rows for k in r)))
        w.writeheader()
        w.writerows(rows)


cdump([{"dataset": ds, "generator": g, "rule": n, **{f"v1_{k}": a1[k] for k in ("N", "E", "A", "W")},
        **{f"v2_{k}": a2[k] for k in ("N", "E", "A", "W")}} for ds, g, n, a1, a2 in MOVED], "arms_cells_that_move.csv")
cdump(SWEEP, "k_sweep_m4.csv")
cdump([{"reader": f"v{v}", **{k: r.get(k) for k in r}} for v, T in ((1, T1), (2, T2)) for r in T["RESULTS"]], "tost_v1_v2.csv")
cdump([{"reader": f"v{v}", "dataset": ds, "qwen_state": a, **{f"pro_{b}": T["JOINT"][ds]["table"][f"{a}->{b}"] for b in ("AC", "AW", "RC", "RW")}}
       for v, T in ((1, T1), (2, T2)) for ds in DATASETS for a in ("AC", "AW", "RC", "RW")], "transitions_v1_v2.csv")
cdump(XT, "screen_crosstabs_v1_v2.csv")
cdump([{k: (json.dumps(v_) if isinstance(v_, list) else v_) for k, v_ in s.items()} for s in SC], "scorer_check.csv")
cdump(STORED, "gsm_hard_stored_precision.csv")
jdump({"changed_rows": [{"generator": g, "dataset": d, "pid": p, "v1": str(STATES[(g, d)][p]["e1"]), "v2": str(STATES[(g, d)][p]["e2"]),
                         "gold": STATES[(g, d)][p]["gold_str"], "correct_v1": STATES[(g, d)][p]["ok1"], "correct_v2": STATES[(g, d)][p]["ok2"],
                         "cert_v1": STATES[(g, d)][p]["cert1"], "cert_v2": STATES[(g, d)][p]["cert2"]} for g, d, p in CHANGED],
       "verify_detail": REVERIFY}, "v2_reverified_rows.json")
jdump({"run": RUN_TS, "stage2_lib_sha256_lf": s2_sha, "reader": v2.VERSION, "sensitivity_scorer": msp.VERSION,
       "z3": z3.get_version_string(),
       "python": sys.version.split()[0], "gates": "A-G passed", "n_changed_rows": len(CHANGED),
       "crc": {f"{ds}|{a}|v{v}": c for (ds, a, v), c in CRC.items()},
       "exact": {f"v{v}|{g}": e for (v, g), e in EXACT.items()}, "integer_targets": {f"v{v}|{g}": n for (v, g), n in NNI.items()},
       "scorers_gsm_hard_k3": {f"v{v}|{g}": s for (v, g), s in SCORERS.items()}}, "summary.json")

# Headline values from the container run of 22 Oct 2026 (same stage2_lib, Z3 5.3.1).
EXPECTED = {"changed rows": 107,
            "Pro GSM-Hard k>=3 v2 A/W": (421, 41), "Pro GSM-Hard wrong v2": 247, "Flash-Lite GSM-Hard wrong v2": 313,
            "TOST GSM-Hard v2 (lo, hi, p5)": (-1.96, 5.06, 0.053),
            "Screen B Pro v2": (26, 41, 107, 206), "Table 4 GSM8K test v2 (k1, k2)": ((510, 16), (367, 9)),
            "exact-rational Pro v2": (69, 421), "S3 k>=3 v2 (qwen, flashlite, pro)": (27, 32, 45),
            "stored-precision right answers v2 (qwen, flashlite, pro)": (710, 954, 1058)}
r_gh = next(r for r in T2["RESULTS"] if r["dataset"] == "gsm_hard")
GOT = {"changed rows": len(CHANGED),
       "Pro GSM-Hard k>=3 v2 A/W": (ARMS[("gsm_hard", "k>=3 m4", "pro")][1]["A"], ARMS[("gsm_hard", "k>=3 m4", "pro")][1]["W"]),
       "Pro GSM-Hard wrong v2": ARMS[("gsm_hard", "k>=3 m4", "pro")][1]["E"],
       "Flash-Lite GSM-Hard wrong v2": ARMS[("gsm_hard", "k>=3 m4", "flashlite")][1]["E"],
       "TOST GSM-Hard v2 (lo, hi, p5)": (round(r_gh["ci_lo_pp"], 2), round(r_gh["ci_hi_pp"], 2), round(r_gh["tost_p_5pp"], 3)),
       "Screen B Pro v2": tuple(x[("v2", "B", "pro")][k] for k in ("cw_flagged", "cw", "other_flagged", "other_wrong")),
       "Table 4 GSM8K test v2 (k1, k2)": tuple((CRC[("gsm8k", a, 2)]["test"]["n_covered"], CRC[("gsm8k", a, 2)]["test"]["cw"]) for a in ("0.1", "0.05")),
       "exact-rational Pro v2": (EXACT[(2, "pro")]["W_exact"], EXACT[(2, "pro")]["A"]),
       "S3 k>=3 v2 (qwen, flashlite, pro)": tuple(SCORERS[(2, g)]["S3"] for g in GENS),
       "stored-precision right answers v2 (qwen, flashlite, pro)": tuple(ST[("v2", g, "k>=3 m4")]["right_stored"] for g in GENS)}
for k in EXPECTED:
    print(f"{'same' if GOT[k] == EXPECTED[k] else 'DIFFERENT':<9} {k}: {GOT[k]}" + ("" if GOT[k] == EXPECTED[k] else f" (container run: {EXPECTED[k]})"))
print("\nwrote", OUT)